
<p> <b>CSCE 5300: Introduction to Big data and Data Science<br><br>
SPRING 2025<br><br>
Assignment-2</b></p>


Mohammed Momin Khizar Uddin

[student identifier removed]

Task 2&3

### **Task 2: Spark Rdds (40%)**

#### Question Descriptive


<h4>A) Explain the differences between RDD transformations and actions, and provide at least three instances of typical actions with examples. (10%)</h4>

##### Min. 200 words

Answer here: **RDD Transformations vs. Actions:**

Resilient Distributed Datasets (RDDs) are the fundamental data structures in Apache Spark. There are two kinds of operations on RDDs: transformations and actions.

**Transformations:**

These operations produce new RDDs from existing RDDs.
They are lazily evaluated, i.e., they are not executed immediately. Instead, Spark records the sequence of transformations to be applied.
Transformations create a lineage or dependency graph of RDDs.
Examples include map, filter, flatMap, groupByKey, reduceByKey, and union.

**Actions:**

These operations trigger the execution of RDD lineage and produce a result to the driver program or write data to external storage.
Actions prompt Spark to carry out the outcome of the transformations.
Some examples include collect, count, reduce, take, first, and saveAsTextFile.

RDD Changes (Creating the Strategy):

 map():


 "Double every number."

   data = sc.parallelize([1, 2, 3])
   Data.map(lambda x: x * 2) #transformation filter() = doubled

 "Keep only the even numbers."

   data = sc.parallelize([1, 2, 3, 4, 5, 6])
   Data.filter(lambda x: x % 2 == 0) = evens  #conversion flatMap():

 "Split sentences into words."

   sc.parallelize(["hello world", "spark is fun"), data
   the lambda line: line.split(" ") = words = data.flatMap  #transformation
 RDD Actions: collect(): Executing and Obtaining Outcomes

 "Show all the results."

   outcome = doubled. #action print(result) #output: [2, 4, 6] count(): collect()

 "How many there are?"
   #activity print(count) #output: 3 reduce(): count = evens.count()

 "Add up all the numbers."

   sc.parallelize([1,2,3,4]) = sum_of_numbers. #action print(sum_of_numbers) #output:10 reduce(lambda x,y: x+y)
 Key points:  Actions are about getting outcomes from RDDs, whereas transformations are about producing new RDDs.


#### Question Implementation (30%)

In [ ]:
#Installing PySpark through pip
!pip install pyspark

# Importing the necessary libraries required for this task.
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, when

# Initializing Spark session asper tutorial given.
spark = SparkSession.builder \
    .appName("Movie Rating Spark Analysis") \
    .getOrCreate()

# Load the datasets into spark session from collab.
movies_path = '/content/movies-1.csv'  # Update it with the file location
ratings_path ='/content/ratings-1-1.csv'  # Update it with the file location

movies_df = spark.read.csv(movies_path, header=True)
ratings_df = spark.read.csv(ratings_path, header=True)

#### B) a.	Display the records and count of the Movie table and Rating table.

(Hint: file.count())

In [ ]:
#Type your code here
#This will print the records of movie and raiting table.
movies_df.show()
ratings_df.show()

#This will display count of the Movie table and Rating table.
print(f"Movie count: {movies_df.count()}")
print(f"Ratings count: {ratings_df.count()}")


#### b. Find the number of unique movies in the dataset.

In [ ]:
#Type your code here
#This part of the code helps finding all the unique movies in the dataset.
unique_movies_count = movies_df.select('movieId').distinct().count()
print(f"Number of unique movies: {unique_movies_count}")

#### c. Find the number of users who have given ratings.

In [ ]:
#Type your code here\
#Extracting number of users who have given the ratings.
unique_users_count = ratings_df.select('userId').distinct().count()
print(f"Number of unique users: {unique_users_count}")

#### d. Count the number of ratings that are less than or equal to 2.


In [ ]:
#Type your code here
#Counting the number of ratings that are <= 2.
rating_counts = ratings_df.filter(col('rating') <= 2).count()
print(f"Count of ratings <= 2: {rating_counts}")


e.	Find the total number of ratings and the average rating for all movies.

In [ ]:
#Type your code here
#Total number of ratings and the average rating for all movies.
total_ratings = ratings_df.count()
average_rating = ratings_df.agg({'rating': 'avg'}).collect()[0][0]
print(f"Total number of ratings: {total_ratings}")
print(f"Average rating: {average_rating:.2f}")

### **Task 3: Analysis using Spark RDD (30%)**

#### Implementation (30%)  

#### A) a.	Find the movie titles with maximum number of ratings.
(Hint: Join the two RDDs created in the previous step on the movieId to create a new RDD with the structure (userId, title) and perform aggregation operation.)

In [ ]:
#Type your code here
#Movies with max number of ratings.
ratings_rdd = ratings_df.rdd.map(lambda row: (row['movieId'], 1))
movies_rdd = movies_df.rdd.map(lambda row: (row['movieId'], row['title']))
movie_ratings_rdd = ratings_rdd.reduceByKey(lambda x, y: x + y)
joined_rdd = movie_ratings_rdd.join(movies_rdd)
max_rated_movie = joined_rdd.max(key=lambda x: x[1][0])
print(f"Movie with maximum number of ratings: {max_rated_movie[1][1]} with {max_rated_movie[1][0]} ratings")

#### b. Find the movie with the highest average rating. Display the movie title and its average rating.
(Hint: Use reduce or top)





In [ ]:
#Type your code here
#The movie with the highest average rating and also displaying the movie title and its average ratings. in this.
ratings_avg_rdd = ratings_df.rdd.map(lambda row: (row['movieId'], (float(row['rating']), 1)))
ratings_avg_rdd = ratings_avg_rdd.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))
avg_ratings_rdd = ratings_avg_rdd.map(lambda x: (x[0], x[1][0] / x[1][1]))

highest_avg_movie = avg_ratings_rdd.join(movies_rdd).max(key=lambda x: x[1][0])
print(f"Movie with highest average rating: {highest_avg_movie[1][1]} with average rating {highest_avg_movie[1][0]:.2f}")


####  c.	Find the top 5 most active users who have given the highest number of ratings.
(Hint: Aggregate the count of ratings per user and sort in descending order.)

In [ ]:
#Type your code here
#Top 5 most active users who have given the highest number of ratings.
active_users_rdd = ratings_df.rdd.map(lambda row: (row['userId'], 1)).reduceByKey(lambda x, y: x + y)
top_5_users = active_users_rdd.takeOrdered(5, key=lambda x: -x[1])
print("Top 5 most active users:")
for user in top_5_users:
    print(f"User {user[0]} with {user[1]} ratings")